## Simple Multi AI Agent Architecture

In [4]:
import os
from typing import TypedDict, Annotated, List, Literal
from langchain_core.messages import (BaseMessage, AIMessage, HumanMessage, SystemMessage)
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph, END, START, MessageGraph
from langgraph.prebuilt import create_react_agent, ToolNode
from langgraph.checkpoint.memory import MemorySaver

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
## Define the state
class AgentState(MessageGraph):
    next_agent:str #Which agent should go next

In [7]:
## Create simple tools
@tool
def search_web(query:str)->str:
    """Search the web for information."""
    #Using Tavily for web search
    search = TavilySearchResults(max_results=3)
    results = search.invoke(query)
    return str(results)

@tool
def write_summary(content:str)->str:
    """Write a summary of the provided content."""
    #Simple summary generation
    summary =f"Summary of finding:\n\n{content[:500]}..."
    return summary

In [12]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm=ChatGroq(model="qwen/qwen3.8-27b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'Qwen3.8 27B', 'status': 'beta', 'release_date': '2026-08-14', 'last_updated': '2026-08-14', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002661E7E7A10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002661EAA4590>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [13]:
## Defing agent functions(Simpler apporch)
def researcher_agent(state:AgentState):
    """Researcher agent that searches for infromation"""

    messages = state["messages"]

    # Add system message for context
    system_msg = SystemMessage(content="You are a research assistant. Use the search_web tool to find information about the user's request.")

    #call LLm with tools
    resarcher_llm = llm.bind_tools([search_web])
    response = resarcher_llm.invoke([system_msg]+ messages)

    #Returning the response and route to writer
    return {
        "messages": [response],
        "next_agent": "writer"
    }

In [14]:
def writer_agent(state:AgentState):
    """Writer agent that creates summeries."""

    messages =state["messages"]
    system_msg =SystemMessage(content="You are a technical writer. review the conversion and create a clear, concise summary of the finding.")

    response = llm.invoke([system_msg] +messages)

    return {
        "messages":[response],
        "next_agent":"end"
    }

In [15]:
#tool excuter node
def excute_tools(state:AgentState):
    """Excute any pending tool calls"""
    messages = state["messages"]
    last_message = messages[-1]

    #check if there are tool calls to excute
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        #create tool node and execute
        tool_node =ToolNode([search_web, write_summary])
        response = tool_node.invoke(state)
        return response
    
    #no tools to excute
    return state